# Scenario Playground Step 12: Purged Walk-Forward Validation

Objective: decide whether Ridge or the shallow Step 11 XGBoost prototype is the more trustworthy S&P 500 methodology after repeated out-of-sample evaluation.

Scope: use the frozen event-aware 584-row, 58-feature matrix; evaluate only the 464 retained pre-final-test rows; preserve target-end-date purging; and never parse the 114 locked final-test targets. This notebook does not train the other 11 targets or export production models.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.walk_forward_validation import (
    build_walk_forward_artifacts,
    persist_walk_forward_artifacts,
)

artifacts = build_walk_forward_artifacts()
output_paths = persist_walk_forward_artifacts(artifacts)
summary = artifacts["summary"]
{
    "python_executable": sys.executable,
    "training_gate": summary["training_gate"],
    "output_paths": output_paths,
}


## Fold construction and leakage protection

Five expanding folds each validate on the next 60 event-aware rows. Before every outer and inner validation period, training rows are removed whenever their 21-session `target_end_date` reaches into the validation period. This is a trading-session target purge, not a count of 21 event-aware samples.

In [ ]:
folds = artifacts["fold_manifest"]
display(folds)
display(pd.Series(summary["matrix_guard"], name="value").to_frame())
assert folds["training_rows"].tolist() == [160, 222, 280, 342, 401]
assert folds["outer_purge_rows"].tolist() == [4, 2, 4, 2, 3]
assert summary["matrix_rows_declared"] == 584
assert summary["pretest_rows_evaluated"] == 464
assert summary["feature_count"] == 58
assert summary["matrix_guard"]["locked_final_test_target_rows_parsed"] == 0


## Repeated out-of-sample results

Each Ridge and XGBoost history variant produces 300 outer-fold predictions. The same zero, fold-training-mean, and trailing-21-observation naive baselines are evaluated on the identical rows. Lower MAE and RMSE are better; positive out-of-sample R-squared is better than predicting the pooled mean.

In [ ]:
aggregate = artifacts["aggregate_metrics"]
display(aggregate[[
    "variant", "sample_count", "mae", "rmse", "oos_r_squared",
    "prediction_actual_correlation", "directional_accuracy",
    "fold_mae_std", "worst_fold_mae", "folds_won_by_mae",
]].round(4))
fold_mae = artifacts["fold_metrics"].pivot(index="fold", columns="variant", values="mae")
display(fold_mae.round(4))
assert artifacts["fold_predictions"].groupby("variant").size().eq(300).all()


## Full history versus recent and recency-weighted history

The recent-five-year Ridge has the lowest pooled MAE, but its advantage over full-history Ridge is under 1%. Full-history Ridge wins more folds and has lower fold dispersion and a better worst fold. The stability tie-break therefore selects full history. Recency weighting does not add reliable value.

In [ ]:
comparison = artifacts["methodology_comparison"]
display(comparison[[
    "variant", "mae", "rmse", "directional_accuracy",
    "fold_mae_std", "worst_fold_mae", "folds_won_by_mae",
    "responsive_inputs_any_fold", "responsive_inputs_all_folds",
]].round(4))
overfit = artifacts["overfit_diagnostics"].groupby("variant", as_index=False)[[
    "training_mae", "inner_validation_mae", "outer_validation_mae",
    "outer_minus_training_mae",
]].mean()
display(overfit.round(4))


## VIX-regime behavior

Fixed VIX thresholds provide a simple regime diagnostic. These rows remain genuine fold-level out-of-sample predictions, but the high-VIX group is small, so its result is informative rather than conclusive.

In [ ]:
regimes = artifacts["regime_metrics"]
selected_regimes = regimes.loc[regimes["variant"].isin([
    "naive_training_mean", "ridge_full_history", "ridge_recent_5y",
    "xgboost_full_history",
])]
display(selected_regimes[[
    "variant", "vix_regime", "sample_count", "mae", "rmse",
    "oos_r_squared", "directional_accuracy",
]].round(4))


## Scenario-shock responsiveness

For each fold and model variant, three real training states are selected near the 10th, 50th, and 90th percentiles of VIX. All eight shocks are reset to neutral zero, then one shock is swept across that fold's observed training range. A response span of at least 0.05 S&P return percentage points is treated as meaningful for this diagnostic.

Ridge responds to every contract input in at least one fold, although Fed Funds and GDP response directions change across folds. The shallow XGBoost variants continue to ignore seven inputs and react meaningfully only to VIX. Neither family crosses the 0.05-point threshold for any shock in the newest fold, so this step selects a family for Step 13 but does not approve a production-ready scenario model.

In [ ]:
sensitivity = artifacts["sensitivity_summary"]
selected_sensitivity = sensitivity.loc[sensitivity["variant"].isin([
    "ridge_full_history", "xgboost_full_history",
])]
display(selected_sensitivity[[
    "variant", "contract_input", "folds_with_meaningful_response",
    "median_prediction_span", "maximum_prediction_span",
    "meaningful_directions_observed",
    "direction_consistent_when_meaningful", "all_sensible",
]].round(4))
ridge_inputs = selected_sensitivity.loc[
    (selected_sensitivity["variant"] == "ridge_full_history")
    & (selected_sensitivity["folds_with_meaningful_response"] > 0),
    "contract_input",
]
xgboost_inputs = selected_sensitivity.loc[
    (selected_sensitivity["variant"] == "xgboost_full_history")
    & (selected_sensitivity["folds_with_meaningful_response"] > 0),
    "contract_input",
]
assert len(set(ridge_inputs)) == 8
assert set(xgboost_inputs) == {"vix_change_points"}


## Step 12 decision

Approve full-history Ridge as the S&P methodology for Step 13. Do not promote XGBoost: its one-period Step 11 MAE win does not repeat, its generalization gap is larger, and it remains unsuitable for an eight-input scenario interface. Ridge's shock coefficients are still predictive associations rather than causal estimates, no input is meaningfully responsive in all five folds, and no shock crosses the audit threshold in the newest fold. This remains a production-inference blocker, not a blocker to Step 13 all-target training.

In [ ]:
checks = pd.Series(summary["validations"], name="passed")
display(checks.to_frame())
assert checks.all()
assert summary["approved_variant"] == "ridge_full_history"
assert summary["approved_history_window_strategy"] == "full_history"
assert summary["models_proceeding_to_step_13"] == ["ridge"]
assert not summary["step_11_xgboost_promotion_confirmed"]
assert summary["training_gate"] == "STEP_12_WALK_FORWARD_COMPLETE_READY_FOR_STEP_13_ALL_TARGETS"
print(summary["methodology_decision"])
print(summary["scenario_responsiveness_limitation"])
print(summary["training_gate"])
